# Controlled post-processing OOF robustness evaluation

This notebook reproduces the controlled post-processing analysis reported in the paper.

- Existing **patience-4 checkpoints** are evaluated without retraining.
- Each sample is predicted only by the fold model for which it was held out.
- Test-time augmentation is disabled.
- The original inference-head fusion rule is retained for each objective.
- Post-processing is deterministic and is applied before the standard resize and normalization.
- The four evaluated objectives are CE-only, ArcFace-only, CE+ArcFace 60:40, and CE+ArcFace 20:80.
- These analyses were conducted after the challenge and did not affect the official sixth-place ranking.

The notebook is a cleaned, repository-oriented version of the original experiment notebook. Duplicate evaluation cells, local comparison logic, unused transformations, and absolute paths were removed without intentionally changing the reported evaluation procedure.

## 1. Imports

The analysis uses the same ConvNeXtV2 and ArcFace definitions as the diagnostic training notebooks. `USE_AMP` remains disabled to match the controlled evaluation.

In [1]:
import gc
import json
import math
import os
import random
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from IPython.display import display
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

## 2. Evaluation configuration

The settings below reproduce the controlled OOF evaluation: five stratified folds, batch size 32, deterministic execution, no TTA, and no mixed-precision inference.

In [2]:
@dataclass(frozen=True)
class EvaluationConfig:
    seed: int = 42
    n_splits: int = 5
    num_classes: int = 10
    batch_size: int = 32
    num_workers: int = 0
    use_amp: bool = False

    data_dir_name: str = "Data"
    train_csv_name: str = "training.csv"

    artifact_dir_name: str = (
        "ensemble_artifacts/diagnostics/postprocessing_robustness"
    )
    result_dir_name: str = "results/diagnostics"

    mean: tuple = (0.485, 0.456, 0.406)
    std: tuple = (0.229, 0.224, 0.225)


CFG = EvaluationConfig()
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Mixed precision:", CFG.use_amp)
print("TTA: disabled")

Device: cuda
Mixed precision: False
TTA: disabled


## 3. Repository paths and deterministic settings

The repository root is located automatically by searching upward for `README.md` and the `Data` directory. Large probability arrays and sample-level records are written under `ensemble_artifacts`, while paper-facing tables are written under `results/diagnostics`.

In [3]:
def find_repository_root(start_path):
    current = Path(start_path).resolve()

    while True:
        has_readme = (current / "README.md").exists()
        has_data = (current / CFG.data_dir_name).exists()

        if has_readme and has_data:
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Repository root could not be located. "
        "Run this notebook inside the cloned repository."
    )


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


REPO_ROOT = find_repository_root(Path.cwd())
DATA_ROOT = REPO_ROOT / CFG.data_dir_name
TRAIN_CSV = DATA_ROOT / CFG.train_csv_name

ARTIFACT_DIR = REPO_ROOT / CFG.artifact_dir_name
PROB_DIR = ARTIFACT_DIR / "probabilities"
SAMPLE_DIR = ARTIFACT_DIR / "samples"
RESULT_DIR = REPO_ROOT / CFG.result_dir_name

for directory in [
    ARTIFACT_DIR,
    PROB_DIR,
    SAMPLE_DIR,
    RESULT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

set_seed(CFG.seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )
except Exception:
    pass

print("Repository root :", REPO_ROOT)
print("Training CSV    :", TRAIN_CSV)
print("Artifact output :", ARTIFACT_DIR)
print("Public results  :", RESULT_DIR)

Repository root : D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution
Training CSV    : D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution\Data\training.csv
Artifact output : D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution\ensemble_artifacts\diagnostics\postprocessing_robustness
Public results  : D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution\results\diagnostics


## 4. Load the training data and recreate the folds

The fold assignment must match the diagnostic training notebooks: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.

In [4]:
if not TRAIN_CSV.exists():
    raise FileNotFoundError(
        f"Training CSV was not found: {TRAIN_CSV}"
    )

train_df = pd.read_csv(TRAIN_CSV)

if "label" in train_df.columns:
    train_df["label"] = train_df["label"].astype(int)
elif "y" in train_df.columns:
    train_df["label"] = train_df["y"].astype(int)
elif "TARGET" in train_df.columns:
    train_df["label"] = train_df["TARGET"].astype(int)
else:
    raise ValueError(
        "training.csv must contain 'label', 'y', or 'TARGET'."
    )

if "ID" not in train_df.columns:
    train_df["ID"] = np.arange(
        len(train_df),
        dtype=int,
    )

path_candidates = [
    "filepath",
    "path",
    "image_path",
    "file_path",
]

path_column = next(
    (
        column
        for column in path_candidates
        if column in train_df.columns
    ),
    None,
)

if path_column is None:
    raise ValueError(
        "training.csv must contain one of: "
        f"{path_candidates}"
    )


def make_absolute_path(path_value):
    path = Path(str(path_value))

    if path.is_absolute():
        return str(path)

    return str(REPO_ROOT / path)


train_df["filepath"] = train_df[path_column].apply(
    make_absolute_path
)

folds = np.full(
    len(train_df),
    -1,
    dtype=int,
)

splitter = StratifiedKFold(
    n_splits=CFG.n_splits,
    shuffle=True,
    random_state=CFG.seed,
)

for fold, (_, valid_indices) in enumerate(
    splitter.split(
        train_df,
        train_df["label"],
    )
):
    folds[valid_indices] = fold

if np.any(folds < 0):
    raise RuntimeError(
        "Some rows were not assigned to a validation fold."
    )

train_df["fold"] = folds

print("Training shape:", train_df.shape)
print("\nClass counts:")
display(
    train_df["label"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

print("Fold counts:")
display(
    train_df["fold"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

Training shape: (7000, 6)

Class counts:


,count
label,
0,700
1,700
2,700
3,700
4,700
5,700
6,700
7,700
8,700


Fold counts:


,count
fold,
0,1400
1,1400
2,1400
3,1400
4,1400


## 5. ArcFace model definition

All four checkpoints use the same ConvNeXtV2 backbone, 512-dimensional neck, CE head, and ArcFace head. At inference time, the ArcFace head receives no labels and therefore returns margin-free scaled cosine logits.

In [5]:
class ArcMarginProduct(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        s=30.0,
        m=0.30,
    ):
        super().__init__()

        self.s = s
        self.m = m

        self.weight = nn.Parameter(
            torch.empty(
                out_features,
                in_features,
            )
        )
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels=None):
        cosine = F.linear(
            F.normalize(embeddings),
            F.normalize(self.weight),
        ).clamp(-1.0, 1.0)

        if labels is None:
            return cosine * self.s

        sine = torch.sqrt(
            torch.clamp(
                1.0 - cosine.pow(2),
                min=1e-7,
            )
        )

        phi = (
            cosine * self.cos_m
            - sine * self.sin_m
        )

        phi = torch.where(
            cosine > self.th,
            phi,
            cosine - self.mm,
        )

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(
            1,
            labels.view(-1, 1).long(),
            1.0,
        )

        logits = (
            one_hot * phi
            + (1.0 - one_hot) * cosine
        )

        return logits * self.s


class ArcFaceModel(nn.Module):
    def __init__(
        self,
        model_name,
        num_classes,
        embedding_dim=512,
        arc_s=30.0,
        arc_m=0.30,
        dropout=0.2,
        pretrained=False,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )

        backbone_output_dim = self.backbone.num_features

        self.neck = nn.Sequential(
            nn.Linear(
                backbone_output_dim,
                embedding_dim,
            ),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU(),
        )

        self.dropout = nn.Dropout(dropout)
        self.ce_head = nn.Linear(
            embedding_dim,
            num_classes,
        )

        self.arc_head = ArcMarginProduct(
            embedding_dim,
            num_classes,
            s=arc_s,
            m=arc_m,
        )

    def forward(self, images, labels=None):
        features = self.backbone(images)
        embeddings = self.neck(features)
        embeddings = self.dropout(embeddings)

        ce_logits = self.ce_head(embeddings)
        arc_logits = self.arc_head(
            embeddings,
            labels,
        )

        return ce_logits, arc_logits, embeddings

## 6. Model specifications and checkpoint locations

`inference_ce_weight` and `inference_arc_weight` are logit-fusion weights. They are not the training-loss weights.

- CE-only: CE logits only.
- ArcFace-only: margin-free ArcFace logits only.
- CE+ArcFace 60:40: equal-weight CE/ArcFace logit fusion.
- CE+ArcFace 20:80: equal-weight CE/ArcFace logit fusion.

In [6]:
@dataclass(frozen=True)
class ModelSpec:
    display_name: str
    checkpoint_directory: Path
    checkpoint_pattern: str
    training_objective: str

    model_name: str = (
        "timm/"
        "convnextv2_base.fcmae_ft_in22k_in1k_384"
    )
    image_size: int = 384
    embedding_dim: int = 512
    dropout: float = 0.2
    arc_s: float = 30.0
    arc_m: float = 0.30

    inference_ce_weight: float = 0.5
    inference_arc_weight: float = 0.5


OFFICIAL_CHECKPOINT_DIR = (
    REPO_ROOT / "ensemble_artifacts"
)
DIAGNOSTIC_CHECKPOINT_DIR = (
    REPO_ROOT
    / "ensemble_artifacts"
    / "diagnostics"
)

MODEL_SPECS = {
    "ce_only": ModelSpec(
        display_name="CE-only",
        checkpoint_directory=(
            DIAGNOSTIC_CHECKPOINT_DIR
        ),
        checkpoint_pattern=(
            "arcface_base_ce_only_"
            "best_fold{fold}.pth"
        ),
        training_objective="CE 100%, ArcFace 0%",
        inference_ce_weight=1.0,
        inference_arc_weight=0.0,
    ),
    "arcface_only": ModelSpec(
        display_name="ArcFace-only",
        checkpoint_directory=(
            DIAGNOSTIC_CHECKPOINT_DIR
        ),
        checkpoint_pattern=(
            "arcface_base_arcface_only_"
            "best_fold{fold}.pth"
        ),
        training_objective="CE 0%, ArcFace 100%",
        inference_ce_weight=0.0,
        inference_arc_weight=1.0,
    ),
    "ce_arc_60_40": ModelSpec(
        display_name="CE+ArcFace 60:40",
        checkpoint_directory=(
            OFFICIAL_CHECKPOINT_DIR
        ),
        checkpoint_pattern=(
            "arcface_base_best_fold{fold}.pth"
        ),
        training_objective="CE 60%, ArcFace 40%",
        inference_ce_weight=0.5,
        inference_arc_weight=0.5,
    ),
    "ce_arc_20_80": ModelSpec(
        display_name="CE+ArcFace 20:80",
        checkpoint_directory=(
            DIAGNOSTIC_CHECKPOINT_DIR
        ),
        checkpoint_pattern=(
            "arcface_base_ce20arc80_"
            "best_fold{fold}.pth"
        ),
        training_objective="CE 20%, ArcFace 80%",
        inference_ce_weight=0.5,
        inference_arc_weight=0.5,
    ),
}


def validate_model_spec(model_key, spec):
    total_weight = (
        spec.inference_ce_weight
        + spec.inference_arc_weight
    )

    if not np.isclose(total_weight, 1.0):
        raise ValueError(
            f"Inference weights for {model_key} "
            "must sum to 1."
        )


def checkpoint_path(spec, fold):
    return (
        spec.checkpoint_directory
        / spec.checkpoint_pattern.format(
            fold=fold,
        )
    )


def build_model(spec):
    return ArcFaceModel(
        model_name=spec.model_name,
        num_classes=CFG.num_classes,
        embedding_dim=spec.embedding_dim,
        arc_s=spec.arc_s,
        arc_m=spec.arc_m,
        dropout=spec.dropout,
        pretrained=False,
    )


def load_model_state(model, checkpoint):
    state = torch.load(
        checkpoint,
        map_location="cpu",
    )

    if isinstance(state, dict):
        if "model_state_dict" in state:
            state = state["model_state_dict"]
        elif "state_dict" in state:
            state = state["state_dict"]

    if any(
        key.startswith("module.")
        for key in state.keys()
    ):
        state = {
            key.replace("module.", "", 1): value
            for key, value in state.items()
        }

    model.load_state_dict(
        state,
        strict=True,
    )

    return model


for model_key, model_spec in MODEL_SPECS.items():
    validate_model_spec(
        model_key,
        model_spec,
    )

## 7. Verify all checkpoints before inference

The analysis stops before loading image data if any fold checkpoint is missing.

In [7]:
missing_checkpoints = []

for model_key, model_spec in MODEL_SPECS.items():
    print("\n" + "=" * 80)
    print(
        f"{model_spec.display_name} "
        f"({model_key})"
    )
    print("=" * 80)

    for fold in range(CFG.n_splits):
        path = checkpoint_path(
            model_spec,
            fold,
        )
        exists = path.exists()

        print(
            f"fold {fold}: {path} "
            f"| exists={exists}"
        )

        if not exists:
            missing_checkpoints.append(path)

if missing_checkpoints:
    missing_text = "\n".join(
        f"- {path}"
        for path in missing_checkpoints
    )
    raise FileNotFoundError(
        "Some required checkpoints were not found:\n"
        f"{missing_text}"
    )

print("\nAll checkpoints were found.")


CE-only (ce_only)
fold 0: D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution\ensemble_artifacts\diagnostics\arcface_base_ce_only_best_fold0.pth | exists=True
fold 1: D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution\ensemble_artifacts\diagnostics\arcface_base_ce_only_best_fold1.pth | exists=True
fold 2: D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution\ensemble_artifacts\diagnostics\arcface_base_ce_only_best_fold2.pth | exists=True
fold 3: D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution\ensemble_artifacts\diagnostics\arcface_base_ce_only_best_fold3.pth | exists=True
fold 4: D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution\ensemble_artifacts\diagnostics\arcface_base_ce_only_

## 8. Deterministic post-processing functions

Only the four transformations reported in the paper are retained.

- `jpg70`: in-memory JPEG encode/decode at quality 70.
- `blur12`: Gaussian blur with σ=1.2, a 5×5 kernel, and reflected boundary padding.
- `down50`: 50% area downsampling followed by cubic restoration.
- `crop80`: 80% center crop followed by cubic restoration.

In [8]:
CONDITIONS = [
    "clean",
    "jpg70",
    "blur12",
    "down50",
    "crop80",
]


def jpeg_compress_rgb(image, quality=70):
    bgr_image = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2BGR,
    )

    success, encoded = cv2.imencode(
        ".jpg",
        bgr_image,
        [
            int(cv2.IMWRITE_JPEG_QUALITY),
            int(quality),
        ],
    )

    if not success:
        raise RuntimeError(
            "JPEG encoding failed."
        )

    decoded = cv2.imdecode(
        encoded,
        cv2.IMREAD_COLOR,
    )

    if decoded is None:
        raise RuntimeError(
            "JPEG decoding failed."
        )

    return cv2.cvtColor(
        decoded,
        cv2.COLOR_BGR2RGB,
    )


def gaussian_blur_rgb(image, sigma=1.2):
    kernel_size = (
        int(round(sigma * 4)) | 1
    )
    kernel_size = max(
        kernel_size,
        3,
    )

    return cv2.GaussianBlur(
        image,
        (kernel_size, kernel_size),
        sigmaX=sigma,
        sigmaY=sigma,
        borderType=cv2.BORDER_REFLECT_101,
    )


def downscale_restore_rgb(
    image,
    scale=0.5,
):
    height, width = image.shape[:2]

    scaled_width = max(
        1,
        int(round(width * scale)),
    )
    scaled_height = max(
        1,
        int(round(height * scale)),
    )

    downscaled = cv2.resize(
        image,
        (scaled_width, scaled_height),
        interpolation=cv2.INTER_AREA,
    )

    return cv2.resize(
        downscaled,
        (width, height),
        interpolation=cv2.INTER_CUBIC,
    )


def center_crop_restore_rgb(
    image,
    crop_ratio=0.8,
):
    height, width = image.shape[:2]

    crop_height = max(
        1,
        int(round(height * crop_ratio)),
    )
    crop_width = max(
        1,
        int(round(width * crop_ratio)),
    )

    top = (height - crop_height) // 2
    left = (width - crop_width) // 2

    cropped = image[
        top:top + crop_height,
        left:left + crop_width,
    ]

    return cv2.resize(
        cropped,
        (width, height),
        interpolation=cv2.INTER_CUBIC,
    )


def apply_postprocess(
    image,
    condition,
):
    if condition == "clean":
        return image

    if condition == "jpg70":
        return jpeg_compress_rgb(
            image,
            quality=70,
        )

    if condition == "blur12":
        return gaussian_blur_rgb(
            image,
            sigma=1.2,
        )

    if condition == "down50":
        return downscale_restore_rgb(
            image,
            scale=0.5,
        )

    if condition == "crop80":
        return center_crop_restore_rgb(
            image,
            crop_ratio=0.8,
        )

    raise ValueError(
        f"Unknown condition: {condition}"
    )

## 9. Evaluation transform and dataset

Each post-processing operation is applied to the original RGB image before resizing to 384×384 and applying ImageNet normalization.

In [9]:
def get_evaluation_transform(image_size):
    return A.Compose([
        A.Resize(
            height=image_size,
            width=image_size,
        ),
        A.Normalize(
            mean=CFG.mean,
            std=CFG.std,
        ),
        ToTensorV2(),
    ])


class RobustnessDataset(Dataset):
    def __init__(
        self,
        dataframe,
        condition,
        transform,
    ):
        self.dataframe = dataframe.reset_index(
            drop=True
        )
        self.condition = condition
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = cv2.imread(
            row["filepath"]
        )

        if image is None:
            raise FileNotFoundError(
                row["filepath"]
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB,
        )

        image = apply_postprocess(
            image,
            self.condition,
        )

        image = self.transform(
            image=image
        )["image"]

        return (
            image,
            int(row["label"]),
            int(row["ID"]),
        )

## 10. Fold-level inference

For each held-out fold, logits are fused according to the original inference rule and converted to probabilities with a single softmax. No TTA or probability ensembling is applied.

In [10]:
@torch.inference_mode()
def predict_one_fold_condition(
    spec,
    fold,
    validation_dataframe,
    condition,
):
    checkpoint = checkpoint_path(
        spec,
        fold,
    )

    if not checkpoint.exists():
        raise FileNotFoundError(
            str(checkpoint)
        )

    model = build_model(spec)
    model = load_model_state(
        model,
        checkpoint,
    )
    model = model.to(DEVICE)
    model.eval()

    dataset = RobustnessDataset(
        validation_dataframe,
        condition=condition,
        transform=get_evaluation_transform(
            spec.image_size
        ),
    )

    loader = DataLoader(
        dataset,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=False,
    )

    probability_batches = []
    label_batches = []
    id_batches = []

    for images, labels, ids in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        if DEVICE.type == "cuda":
            autocast_context = torch.autocast(
                device_type="cuda",
                enabled=CFG.use_amp,
            )
        else:
            autocast_context = nullcontext()

        with autocast_context:
            (
                ce_logits,
                arc_logits,
                _,
            ) = model(
                images,
                labels=None,
            )

            fused_logits = (
                spec.inference_ce_weight
                * ce_logits
                + spec.inference_arc_weight
                * arc_logits
            )

            probabilities = torch.softmax(
                fused_logits,
                dim=1,
            )

        probability_batches.append(
            probabilities
            .float()
            .cpu()
            .numpy()
        )
        label_batches.append(
            labels.numpy()
        )
        id_batches.append(
            ids.numpy()
        )

    del model, dataset, loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        np.concatenate(
            id_batches,
            axis=0,
        ),
        np.concatenate(
            label_batches,
            axis=0,
        ),
        np.concatenate(
            probability_batches,
            axis=0,
        ),
    )

## 11. OOF evaluation for one model

The held-out probabilities are restored to the original training-row order and saved for every condition.

In [11]:
def run_oof_for_model(
    model_key,
    spec,
    conditions,
):
    print("\n" + "=" * 100)
    print(
        f"MODEL: {spec.display_name} "
        f"({model_key})"
    )
    print("=" * 100)

    probabilities_by_condition = {}
    fold_rows = []

    for condition in conditions:
        print("\n" + "-" * 80)
        print(
            f"{spec.display_name} "
            f"| condition={condition}"
        )
        print("-" * 80)

        oof_probabilities = np.zeros(
            (
                len(train_df),
                CFG.num_classes,
            ),
            dtype=np.float32,
        )

        for fold in range(CFG.n_splits):
            validation_indices = np.where(
                train_df["fold"].to_numpy()
                == fold
            )[0]

            validation_dataframe = (
                train_df
                .iloc[validation_indices]
                .reset_index(drop=True)
            )

            (
                ids,
                labels,
                probabilities,
            ) = predict_one_fold_condition(
                spec=spec,
                fold=fold,
                validation_dataframe=(
                    validation_dataframe
                ),
                condition=condition,
            )

            expected_ids = (
                validation_dataframe["ID"]
                .to_numpy()
            )
            expected_labels = (
                validation_dataframe["label"]
                .to_numpy()
            )

            if not np.array_equal(
                ids,
                expected_ids,
            ):
                raise RuntimeError(
                    "ID order mismatch: "
                    f"model={model_key}, "
                    f"condition={condition}, "
                    f"fold={fold}"
                )

            if not np.array_equal(
                labels,
                expected_labels,
            ):
                raise RuntimeError(
                    "Label order mismatch: "
                    f"model={model_key}, "
                    f"condition={condition}, "
                    f"fold={fold}"
                )

            oof_probabilities[
                validation_indices
            ] = probabilities

            fold_accuracy = accuracy_score(
                labels,
                probabilities.argmax(axis=1),
            )

            fold_rows.append({
                "model": model_key,
                "display_name": spec.display_name,
                "condition": condition,
                "fold": int(fold),
                "n_samples": int(
                    len(validation_indices)
                ),
                "accuracy": float(
                    fold_accuracy
                ),
            })

            print(
                f"fold {fold} "
                f"| n={len(validation_indices)} "
                f"| acc={fold_accuracy:.6f}"
            )

        probabilities_by_condition[
            condition
        ] = oof_probabilities

        probability_path = (
            PROB_DIR
            / f"{model_key}_{condition}.npy"
        )
        np.save(
            probability_path,
            oof_probabilities,
        )

        overall_accuracy = accuracy_score(
            train_df["label"].to_numpy(),
            oof_probabilities.argmax(axis=1),
        )

        print(
            f"{spec.display_name} "
            f"| {condition} "
            f"| OOF acc={overall_accuracy:.6f}"
        )

    fold_dataframe = pd.DataFrame(
        fold_rows
    )

    return (
        probabilities_by_condition,
        fold_dataframe,
    )

## 12. Summary metrics

In addition to accuracy, the notebook records prediction flips relative to clean images, retention of clean-correct predictions, confidence, entropy, and class-wise behavior.

In [12]:
def entropy_from_probabilities(
    probabilities,
):
    return -np.sum(
        probabilities
        * np.log(
            probabilities + 1e-12
        ),
        axis=1,
    )


def top2_margin_from_probabilities(
    probabilities,
):
    sorted_probabilities = -np.sort(
        -probabilities,
        axis=1,
    )

    return (
        sorted_probabilities[:, 0]
        - sorted_probabilities[:, 1]
    )


def summarize_model_results(
    model_key,
    spec,
    probabilities_by_condition,
    conditions,
):
    true_labels = (
        train_df["label"].to_numpy()
    )
    ids = train_df["ID"].to_numpy()
    folds = train_df["fold"].to_numpy()

    if "path" in train_df.columns:
        paths = train_df["path"].to_numpy()
    else:
        paths = (
            train_df["filepath"].to_numpy()
        )

    clean_probabilities = (
        probabilities_by_condition["clean"]
    )
    clean_predictions = (
        clean_probabilities.argmax(axis=1)
    )
    clean_correct = (
        clean_predictions == true_labels
    )
    clean_accuracy = accuracy_score(
        true_labels,
        clean_predictions,
    )

    summary_rows = []
    class_rows = []
    sample_frames = []

    for condition in conditions:
        probabilities = (
            probabilities_by_condition[
                condition
            ]
        )
        predictions = probabilities.argmax(
            axis=1
        )
        correct = predictions == true_labels

        accuracy = accuracy_score(
            true_labels,
            predictions,
        )
        accuracy_drop = (
            clean_accuracy - accuracy
        )

        flipped = (
            predictions != clean_predictions
        )
        lost = (
            clean_correct & ~correct
        )
        gained = (
            ~clean_correct & correct
        )

        if clean_correct.sum() > 0:
            retention = float(
                np.sum(
                    clean_correct & correct
                )
                / clean_correct.sum()
            )
        else:
            retention = np.nan

        confidence = probabilities.max(
            axis=1
        )
        entropy = (
            entropy_from_probabilities(
                probabilities
            )
        )
        top2_margin = (
            top2_margin_from_probabilities(
                probabilities
            )
        )

        summary_rows.append({
            "model": model_key,
            "display_name": spec.display_name,
            "condition": condition,
            "accuracy": float(accuracy),
            "clean_accuracy": float(
                clean_accuracy
            ),
            "accuracy_drop": float(
                accuracy_drop
            ),
            "flip_rate": float(
                np.mean(flipped)
            ),
            "retention_clean_correct": (
                retention
            ),
            "lost_clean_correct": int(
                lost.sum()
            ),
            "gained_from_clean_wrong": int(
                gained.sum()
            ),
            "n_correct": int(
                correct.sum()
            ),
            "n_samples": int(
                len(true_labels)
            ),
            "mean_confidence": float(
                confidence.mean()
            ),
            "mean_entropy": float(
                entropy.mean()
            ),
            "mean_top2_margin": float(
                top2_margin.mean()
            ),
        })

        for class_id in range(
            CFG.num_classes
        ):
            class_mask = (
                true_labels == class_id
            )

            class_accuracy = accuracy_score(
                true_labels[class_mask],
                predictions[class_mask],
            )
            clean_class_accuracy = (
                accuracy_score(
                    true_labels[class_mask],
                    clean_predictions[
                        class_mask
                    ],
                )
            )

            class_rows.append({
                "model": model_key,
                "display_name": (
                    spec.display_name
                ),
                "condition": condition,
                "class_id": int(class_id),
                "accuracy": float(
                    class_accuracy
                ),
                "clean_accuracy": float(
                    clean_class_accuracy
                ),
                "accuracy_drop": float(
                    clean_class_accuracy
                    - class_accuracy
                ),
                "n_samples": int(
                    class_mask.sum()
                ),
                "flip_rate": float(
                    np.mean(
                        flipped[class_mask]
                    )
                ),
                "lost_clean_correct": int(
                    lost[class_mask].sum()
                ),
                "gained_from_clean_wrong": (
                    int(
                        gained[
                            class_mask
                        ].sum()
                    )
                ),
            })

        sample_frames.append(
            pd.DataFrame({
                "ID": ids,
                "fold": folds,
                "label": true_labels,
                "path": paths,
                "model": model_key,
                "display_name": (
                    spec.display_name
                ),
                "condition": condition,
                "prediction": predictions,
                "correct": correct,
                "confidence": confidence,
                "entropy": entropy,
                "top2_margin": top2_margin,
                "clean_prediction": (
                    clean_predictions
                ),
                "clean_correct": (
                    clean_correct
                ),
                "flip_from_clean": flipped,
                "lost_clean_correct": lost,
                "gained_from_clean_wrong": (
                    gained
                ),
            })
        )

    summary_dataframe = pd.DataFrame(
        summary_rows
    )
    class_dataframe = pd.DataFrame(
        class_rows
    )
    sample_dataframe = pd.concat(
        sample_frames,
        axis=0,
        ignore_index=True,
    )

    sample_dataframe.to_csv(
        SAMPLE_DIR
        / f"sample_{model_key}.csv",
        index=False,
    )

    return (
        summary_dataframe,
        class_dataframe,
    )

## 13. Run the controlled evaluation

This cell evaluates all four models under all five conditions. On a single GPU it may take substantial time because 100 fold-condition model loads are required.

In [13]:
all_summary_dataframes = []
all_class_dataframes = []
all_fold_dataframes = []

for model_key, model_spec in MODEL_SPECS.items():
    (
        probabilities_by_condition,
        fold_dataframe,
    ) = run_oof_for_model(
        model_key=model_key,
        spec=model_spec,
        conditions=CONDITIONS,
    )

    (
        summary_dataframe,
        class_dataframe,
    ) = summarize_model_results(
        model_key=model_key,
        spec=model_spec,
        probabilities_by_condition=(
            probabilities_by_condition
        ),
        conditions=CONDITIONS,
    )

    all_summary_dataframes.append(
        summary_dataframe
    )
    all_class_dataframes.append(
        class_dataframe
    )
    all_fold_dataframes.append(
        fold_dataframe
    )

summary_df = pd.concat(
    all_summary_dataframes,
    axis=0,
    ignore_index=True,
)
classwise_df = pd.concat(
    all_class_dataframes,
    axis=0,
    ignore_index=True,
)
fold_results_df = pd.concat(
    all_fold_dataframes,
    axis=0,
    ignore_index=True,
)

summary_path = (
    RESULT_DIR
    / "postprocessing_summary.csv"
)
classwise_path = (
    RESULT_DIR
    / "postprocessing_classwise_accuracy.csv"
)
fold_path = (
    RESULT_DIR
    / "postprocessing_fold_results.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)
classwise_df.to_csv(
    classwise_path,
    index=False,
)
fold_results_df.to_csv(
    fold_path,
    index=False,
)

print("Saved:")
print(summary_path)
print(classwise_path)
print(fold_path)

display(summary_df)


MODEL: CE-only (ce_only)

--------------------------------------------------------------------------------
CE-only | condition=clean
--------------------------------------------------------------------------------
fold 0 | n=1400 | acc=0.988571
fold 1 | n=1400 | acc=0.974286
fold 2 | n=1400 | acc=0.980714
fold 3 | n=1400 | acc=0.995714
fold 4 | n=1400 | acc=0.991429
CE-only | clean | OOF acc=0.986143

--------------------------------------------------------------------------------
CE-only | condition=jpg70
--------------------------------------------------------------------------------
fold 0 | n=1400 | acc=0.980000
fold 1 | n=1400 | acc=0.950000
fold 2 | n=1400 | acc=0.952143
fold 3 | n=1400 | acc=0.985714
fold 4 | n=1400 | acc=0.980714
CE-only | jpg70 | OOF acc=0.969714

--------------------------------------------------------------------------------
CE-only | condition=blur12
--------------------------------------------------------------------------------
fold 0 | n=1400 | acc=0.98

,model,display_name,condition,accuracy,clean_accuracy,accuracy_drop,flip_rate,retention_clean_correct,lost_clean_correct,gained_from_clean_wrong,n_correct,n_samples,mean_confidence,mean_entropy,mean_top2_margin
0,ce_only,CE-only,clean,0.986143,0.986143,0.000000,0.000000,1.000000,0,0,6903,7000,0.969230,0.167750,0.957212
1,ce_only,CE-only,jpg70,0.969714,0.986143,0.016429,0.019571,0.982037,124,9,6788,7000,0.963265,0.185144,0.947004
2,ce_only,CE-only,blur12,0.977571,0.986143,0.008571,0.014571,0.988556,79,19,6843,7000,0.963504,0.186127,0.947869
3,ce_only,CE-only,down50,0.982714,0.986143,0.003429,0.008714,0.994061,41,17,6879,7000,0.965887,0.178508,0.951725
4,ce_only,CE-only,crop80,0.983429,0.986143,0.002714,0.017571,0.989715,71,52,6884,7000,0.966922,0.175389,0.953225
5,arcface_only,ArcFace-only,clean,0.989857,0.989857,0.000000,0.000000,1.000000,0,0,6929,7000,0.999375,0.001785,0.998772
6,arcface_only,ArcFace-only,jpg70,0.975429,0.989857,0.014429,0.018143,0.983836,112,11,6828,7000,0.997788,0.005133,0.995644
7,arcface_only,ArcFace-only,blur12,0.987571,0.989857,0.002286,0.006714,0.995959,28,12,6913,7000,0.998504,0.003659,0.997130
8,arcface_only,ArcFace-only,down50,0.988714,0.989857,0.001143,0.004857,0.997258,19,11,6921,7000,0.998819,0.002931,0.997754
9,arcface_only,ArcFace-only,crop80,0.985714,0.989857,0.004143,0.014143,0.991052,62,33,6900,7000,0.998554,0.003470,0.997225


## 14. Detailed diagnostic tables

These tables expose accuracy drop, prediction flips, and retention under each transformation. The compact paper-facing table is generated in the next section.

In [14]:
condition_order = [
    "clean",
    "jpg70",
    "blur12",
    "down50",
    "crop80",
]

accuracy_table = summary_df.pivot(
    index="display_name",
    columns="condition",
    values="accuracy",
).reindex(columns=condition_order)

drop_table = summary_df.pivot(
    index="display_name",
    columns="condition",
    values="accuracy_drop",
).reindex(columns=condition_order)

flip_table = summary_df.pivot(
    index="display_name",
    columns="condition",
    values="flip_rate",
).reindex(columns=condition_order)

retention_table = summary_df.pivot(
    index="display_name",
    columns="condition",
    values="retention_clean_correct",
).reindex(columns=condition_order)

print("OOF accuracy")
display(accuracy_table)

print("Accuracy drop from clean")
display(drop_table)

print("Prediction flip rate")
display(flip_table)

print("Clean-correct retention")
display(retention_table)

OOF accuracy


condition,clean,jpg70,blur12,down50,crop80
display_name,,,,,
ArcFace-only,0.989857,0.975429,0.987571,0.988714,0.985714
CE+ArcFace 20:80,0.990143,0.979571,0.985286,0.987143,0.987429
CE+ArcFace 60:40,0.989714,0.974429,0.983429,0.987143,0.986286
CE-only,0.986143,0.969714,0.977571,0.982714,0.983429


Accuracy drop from clean


condition,clean,jpg70,blur12,down50,crop80
display_name,,,,,
ArcFace-only,0.0,0.014429,0.002286,0.001143,0.004143
CE+ArcFace 20:80,0.0,0.010571,0.004857,0.003000,0.002714
CE+ArcFace 60:40,0.0,0.015286,0.006286,0.002571,0.003429
CE-only,0.0,0.016429,0.008571,0.003429,0.002714


Prediction flip rate


condition,clean,jpg70,blur12,down50,crop80
display_name,,,,,
ArcFace-only,0.0,0.018143,0.006714,0.004857,0.014143
CE+ArcFace 20:80,0.0,0.014143,0.009286,0.007286,0.014143
CE+ArcFace 60:40,0.0,0.018714,0.009143,0.006000,0.013857
CE-only,0.0,0.019571,0.014571,0.008714,0.017571


Clean-correct retention


condition,clean,jpg70,blur12,down50,crop80
display_name,,,,,
ArcFace-only,1.0,0.983836,0.995959,0.997258,0.991052
CE+ArcFace 20:80,1.0,0.987736,0.993075,0.994950,0.991632
CE+ArcFace 60:40,1.0,0.982968,0.992206,0.995670,0.991339
CE-only,1.0,0.982037,0.988556,0.994061,0.989715


## 15. Paper-facing robustness table

`mean_drop` is the mean clean-to-processed accuracy drop across the four transformations. `worst_accuracy` is the lowest processed-condition accuracy. The resulting CSV corresponds to the robustness table reported in the paper.

In [15]:
processed_conditions = [
    "jpg70",
    "blur12",
    "down50",
    "crop80",
]

paper_rows = []

for model_key, model_spec in MODEL_SPECS.items():
    model_dataframe = summary_df[
        summary_df["model"] == model_key
    ].copy()

    clean_row = model_dataframe[
        model_dataframe["condition"]
        == "clean"
    ].iloc[0]

    processed_dataframe = model_dataframe[
        model_dataframe["condition"].isin(
            processed_conditions
        )
    ]

    accuracy_by_condition = (
        processed_dataframe
        .set_index("condition")["accuracy"]
    )

    paper_rows.append({
        "objective": model_spec.display_name,
        "clean": float(
            clean_row["accuracy"]
        ),
        "jpg70": float(
            accuracy_by_condition["jpg70"]
        ),
        "blur12": float(
            accuracy_by_condition["blur12"]
        ),
        "down50": float(
            accuracy_by_condition["down50"]
        ),
        "crop80": float(
            accuracy_by_condition["crop80"]
        ),
        "mean_drop": float(
            processed_dataframe[
                "accuracy_drop"
            ].mean()
        ),
        "worst_accuracy": float(
            processed_dataframe[
                "accuracy"
            ].min()
        ),
        "mean_flip_rate": float(
            processed_dataframe[
                "flip_rate"
            ].mean()
        ),
        "mean_retention": float(
            processed_dataframe[
                "retention_clean_correct"
            ].mean()
        ),
    })

paper_table_df = pd.DataFrame(
    paper_rows
)

paper_table_path = (
    RESULT_DIR
    / "table2_postprocessing_accuracy.csv"
)
paper_table_df.to_csv(
    paper_table_path,
    index=False,
)

display(paper_table_df)
print("Saved:", paper_table_path)

,objective,clean,jpg70,blur12,down50,crop80,mean_drop,worst_accuracy,mean_flip_rate,mean_retention
0,CE-only,0.986143,0.969714,0.977571,0.982714,0.983429,0.007786,0.969714,0.015107,0.988592
1,ArcFace-only,0.989857,0.975429,0.987571,0.988714,0.985714,0.005500,0.975429,0.010964,0.992026
2,CE+ArcFace 60:40,0.989714,0.974429,0.983429,0.987143,0.986286,0.006893,0.974429,0.011929,0.990546
3,CE+ArcFace 20:80,0.990143,0.979571,0.985286,0.987143,0.987429,0.005286,0.979571,0.011214,0.991848


Saved: D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution\results\diagnostics\table2_postprocessing_accuracy.csv


## 16. Save analysis metadata

The metadata file records the exact transformation definitions, fold rule, checkpoint files, and inference weights used by this cleaned notebook.

In [17]:
#def relative_to_repository(path):
#    try:
#        return str(
#            path.relative_to(REPO_ROOT)
#        )
#    except ValueError:
#        return str(path)

def relative_to_repository(path):
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


metadata = {
    "source_notebook": (
        "Controlled_post-processing_"
        "OOF_robustness_evaluation2(3).ipynb"
    ),
    "evaluation": {
        "seed": CFG.seed,
        "n_splits": CFG.n_splits,
        "fold_rule": (
            "StratifiedKFold("
            "n_splits=5, shuffle=True, "
            "random_state=42)"
        ),
        "oof_only": True,
        "retraining": False,
        "tta": False,
        "mixed_precision": CFG.use_amp,
        "softmax_after_logit_fusion": True,
    },
    "conditions": {
        "clean": "No additional post-processing.",
        "jpg70": (
            "JPEG encode/decode at quality 70."
        ),
        "blur12": (
            "Gaussian blur, sigma=1.2, "
            "5x5 kernel, BORDER_REFLECT_101."
        ),
        "down50": (
            "50% INTER_AREA downscale and "
            "INTER_CUBIC restoration."
        ),
        "crop80": (
            "80% center crop and "
            "INTER_CUBIC restoration."
        ),
    },
    "models": {},
}

for model_key, model_spec in MODEL_SPECS.items():
    metadata["models"][model_key] = {
        "display_name": (
            model_spec.display_name
        ),
        "training_objective": (
            model_spec.training_objective
        ),
        "checkpoint_directory": (
            relative_to_repository(
                model_spec
                .checkpoint_directory
            )
        ),
        "checkpoint_pattern": (
            model_spec.checkpoint_pattern
        ),
        "inference_ce_weight": (
            model_spec
            .inference_ce_weight
        ),
        "inference_arc_weight": (
            model_spec
            .inference_arc_weight
        ),
        "image_size": model_spec.image_size,
        "embedding_dim": (
            model_spec.embedding_dim
        ),
        "arc_s": model_spec.arc_s,
        "arc_m": model_spec.arc_m,
    }

metadata_path = (
    RESULT_DIR
    / "postprocessing_metadata.json"
)

with metadata_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved:", metadata_path)

Saved: D:\work\Notebooks\Kaggle\dlmmdd-workshop-synthetic-source-attribution-challenge\Github\dlmmdd-source-attribution\results\diagnostics\postprocessing_metadata.json


## Expected paper values

After the required checkpoints are placed in the documented directories, the compact table should reproduce the paper values up to the displayed precision:

| Objective | Clean | JPEG70 | Blur | Down50 | Crop80 | Mean drop | Worst |
|---|---:|---:|---:|---:|---:|---:|---:|
| CE-only | 0.9861 | 0.9697 | 0.9776 | 0.9827 | 0.9834 | 0.0078 | 0.9697 |
| ArcFace-only | 0.9899 | 0.9754 | 0.9876 | 0.9887 | 0.9857 | 0.0055 | 0.9754 |
| CE+ArcFace 60:40 | 0.9897 | 0.9744 | 0.9834 | 0.9871 | 0.9863 | 0.0069 | 0.9744 |
| CE+ArcFace 20:80 | 0.9901 | 0.9796 | 0.9853 | 0.9871 | 0.9874 | 0.0053 | 0.9796 |

Small differences beyond the displayed precision should be investigated by checking the fold assignment, checkpoint files, package versions, and image-decoding environment.